In [4]:
# Core libraries
import pandas as pd
import re

# Sklearn modules
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report


step 1:Loading the Dataset

In [5]:
df =pd.read_excel("C:/Users/chand/OneDrive - Indian School of Business/Documents/AI_PROJECT.xlsx")
df.head()

,CODE,SKILL
0,for i in range(5):\n print(i),Beginner
1,a = int(input())\nb = int(input())\nprint(a + b),Beginner
2,"numbers = [1, 2, 3, 4]\nfor n in numbers:\n ...",Beginner
3,a = 5\nb = 2\nprint(a // b),Beginner
4,"num1 = ""5""\nnum2 = 3\nresult = num1 * num2\npr...",Beginner


step2: Checking the Dataset

In [6]:
df.columns

Index(['CODE', 'SKILL'], dtype='object')

In [7]:
df['SKILL'].value_counts()

SKILL
Advanced        41
Intermediate    33
Beginner        28
intermediate     6
beginner         1
Name: count, dtype: int64

Step3:Preprocessing

In [8]:
def clean_code(text):
    text=text.lower()
    text=re.sub(r"\s+", " ", text)  # remove extra spaces
    return text

df['code_clean']=df['CODE'].apply(clean_code)
df.head()

,CODE,SKILL,code_clean
0,for i in range(5):\n print(i),Beginner,for i in range(5): print(i)
1,a = int(input())\nb = int(input())\nprint(a + b),Beginner,a = int(input()) b = int(input()) print(a + b)
2,"numbers = [1, 2, 3, 4]\nfor n in numbers:\n ...",Beginner,"numbers = [1, 2, 3, 4] for n in numbers: print(n)"
3,a = 5\nb = 2\nprint(a // b),Beginner,a = 5 b = 2 print(a // b)
4,"num1 = ""5""\nnum2 = 3\nresult = num1 * num2\npr...",Beginner,"num1 = ""5"" num2 = 3 result = num1 * num2 print..."


Step4:Spliting data into Train data and Test Data

In [9]:
X = df['code_clean']
y = df['SKILL']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

step5:Converting Text to Numbers

In [10]:
vectorizer=TfidfVectorizer(analyzer="char", ngram_range=(2, 5))
X_train_vec=vectorizer.fit_transform(X_train)
X_test_vec=vectorizer.transform(X_test)


In [13]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print("\n======================")
    print(name)
    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred, zero_division=0))

    return model, acc

step 7:Model Training

In [14]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, class_weight="balanced"),
    "SVM": SVC(class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced"),
    "Naive Bayes": MultinomialNB()
}

results = {}
trained_models = {}

for name, model in models.items():
    trained_model, acc = train_and_evaluate(
        model,
        X_train_vec,
        y_train,
        X_test_vec,
        y_test,
        name
    )

    results[name] = acc
    trained_models[name] = trained_model


Logistic Regression
Accuracy: 0.8181818181818182
              precision    recall  f1-score   support

    Advanced       0.83      1.00      0.91        10
    Beginner       0.75      1.00      0.86         6
Intermediate       1.00      0.50      0.67         4
intermediate       0.00      0.00      0.00         2

    accuracy                           0.82        22
   macro avg       0.65      0.62      0.61        22
weighted avg       0.77      0.82      0.77        22


SVM
Accuracy: 0.8636363636363636
              precision    recall  f1-score   support

    Advanced       1.00      0.90      0.95        10
    Beginner       0.75      1.00      0.86         6
Intermediate       0.80      1.00      0.89         4
intermediate       0.00      0.00      0.00         2

    accuracy                           0.86        22
   macro avg       0.64      0.72      0.67        22
weighted avg       0.80      0.86      0.83        22


Random Forest
Accuracy: 0.7272727272727273
  

In [ ]:
Step 8: BEST MODEL SELECTION

In [15]:
best_model_name = max(results, key=results.get)
best_model = trained_models[best_model_name]

print("\n======================")
print("BEST MODEL:", best_model_name)
print("BEST ACCURACY:", results[best_model_name])


BEST MODEL: SVM
BEST ACCURACY: 0.8636363636363636


step9:Testing

In [18]:
def predict_level(code_snippet):
    cleaned = clean_code(code_snippet)
    vec = vectorizer.transform([cleaned])
    return best_model.predict(vec)[0]

In [21]:
user_code=input("Enter your code snippet: ")
print("Predicted Level:", predict_level(user_code))

Enter your code snippet:  num = 10  if num % 2 == 0:     print("Even") else:     print("Odd")


Predicted Level: Beginner


In [22]:
user_code=input("Enter your code snippet: ")
print("Predicted Level:", predict_level(user_code))

Enter your code snippet:  def msg(name):     return f"Hello, {name}!"  # Assigning the function to a variable f = msg  # Calling the function using the variable print(f("Emma"))


Predicted Level: Intermediate


In [25]:
import joblib
joblib.dump(best_model, "code_level_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

step9: creating Streamlit App File

In [25]:
!pip install streamlit

In [26]:
%%writefile app.py
import streamlit as st
import joblib
import re

model = joblib.load("code_level_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

def clean_code(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text

def predict_level(code_snippet):
    cleaned = clean_code(code_snippet)
    vec = vectorizer.transform([cleaned])
    return model.predict(vec)[0]

st.title("🧠 Code Skill Level Predictor")

st.write("Enter a Python code snippet and get predicted skill level")

code_input = st.text_area("Paste your code here:", height=200)

if st.button("Predict"):
    if code_input.strip() == "":
        st.error("Please enter code!")
    else:
        prediction = predict_level(code_input)
        st.success(f"Predicted Level: {prediction}")

Overwriting app.py


In [ ]:
!streamlit run app.py